# TP 2 — Mesurer ce que coûte un format### Module 2 — Stockage distribué · Big Data M2 / Ingénieur**Durée :** 2 heures · **Noté sur 20**---## Ce que vous devez savoir faire à la fin de ce TP1. Écrire le même jeu de données dans cinq formats et **mesurer** l'écart.2. Démontrer expérimentalement qu'un fichier gzip n'est pas splittable.3. Lire un plan d'exécution et y trouver la métrique du volume réellement lu.4. Montrer que le predicate pushdown **ne sert à rien sur des données non triées**.5. Choisir un format par le calcul, et non par habitude.## LivrableCe notebook complété, exporté en HTML, avec le **tableau de synthèse rempli**et les six questions rédigées.## Barème| Exercice | Sujet | Points ||---|---|---|| 1 | Préparation et écriture dans cinq formats | 3 || 2 | Comparaison des tailles | 4 || 3 | Splittabilité | 4 || 4 | Projection et pushdown | 5 || 5 | L'effet du tri | 3 || 6 | Synthèse chiffrée | 1 |## Avant de commencerLe cluster doit tourner (`docker compose up -d`). Ce TP écrit environ 2 Giosur HDFS : vérifiez que vous avez la place.

---# Exercice 1 — Préparer et écrire  *(3 points)***Objectif.** Produire le même jeu de données dans cinq formats, en mesurant le tempsd'écriture de chacun.

In [ ]:
# 1.1 — Session Sparkfrom pyspark.sql import SparkSessionimport time, subprocessUTILISATEUR = "etudiant"        # <-- votre nom de familleFILIERE     = "if"              # "if" = finance, "an" = art numériquespark = (SparkSession.builder         .appName("TP2 - Formats")         .master("local[*]")         .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")         .config("spark.sql.parquet.compression.codec", "snappy")         .getOrCreate())spark.sparkContext.setLogLevel("WARN")print("Spark", spark.version)

In [ ]:
# 1.2 — Générer le jeu de données (3 à 5 minutes)#        4 millions de lignes : assez pour que les écarts soient nets.!python /home/tinku/cours/99-Infra/scripts/generate_datasets.py \        --filiere {FILIERE} --sortie /home/tinku/work/data --evenements 4000000FICHIER = "if_transactions.jsonl" if FILIERE == "if" else "an_evenements.jsonl"!ls -lh /home/tinku/work/data/{FICHIER}

In [ ]:
# 1.3 — Charger en mémoire une fois pour toutes, avec un schéma DÉCLARÉ.#        Déclarer le schéma évite à Spark une passe d'inférence sur tout le fichier.from pyspark.sql.types import (StructType, StructField, StringType,                               DoubleType, BooleanType, LongType)if FILIERE == "if":    schema = StructType([        StructField("id_transaction",   StringType()),        StructField("horodatage",       StringType()),        StructField("id_compte",        StringType()),        StructField("id_marchand",      StringType()),        StructField("montant",          DoubleType()),        StructField("devise",           StringType()),        StructField("canal",            StringType()),        StructField("pays_transaction", StringType()),        StructField("statut",           StringType()),        StructField("est_fraude",       BooleanType()),    ])else:    schema = StructType([        StructField("id_evenement",      StringType()),        StructField("horodatage",        StringType()),        StructField("id_utilisateur",    StringType()),        StructField("id_oeuvre",         StringType()),        StructField("type_evenement",    StringType()),        StructField("position_s",        LongType()),        StructField("appareil",          StringType()),        StructField("pays",              StringType()),        StructField("qualite",           StringType()),        StructField("ms_mise_en_tampon", LongType()),    ])df = spark.read.schema(schema).json(f"file:///home/tinku/work/data/{FICHIER}").cache()print("lignes :", df.count())print("colonnes :", len(df.columns))df.show(3, truncate=False)

In [ ]:
# 1.4 — Écrire dans cinq formats, en chronométrant chacunBASE = f"hdfs:///user/{UTILISATEUR}/tp2"!hdfs dfs -rm -r -f -skipTrash {BASE}temps = {}def ecrire(nom, fn):    t0 = time.time()    fn()    temps[nom] = round(time.time() - t0, 1)    print(f"{nom:22s} {temps[nom]:6.1f} s")ecrire("csv",            lambda: df.write.mode("overwrite").option("header", True)                                   .csv(f"{BASE}/csv"))ecrire("csv.gz",         lambda: df.write.mode("overwrite").option("header", True)                                   .option("compression", "gzip").csv(f"{BASE}/csv_gz"))ecrire("json",           lambda: df.write.mode("overwrite").json(f"{BASE}/json"))ecrire("avro",           lambda: df.write.mode("overwrite").format("avro")                                   .save(f"{BASE}/avro"))ecrire("parquet-snappy", lambda: df.write.mode("overwrite")                                   .option("compression", "snappy")                                   .parquet(f"{BASE}/parquet_snappy"))ecrire("parquet-zstd",   lambda: df.write.mode("overwrite")                                   .option("compression", "zstd")                                   .parquet(f"{BASE}/parquet_zstd"))

---# Exercice 2 — Comparer les tailles  *(4 points)*

In [ ]:
# 2.1 — Relever la taille de chaque répertoireimport redef taille_hdfs(chemin):    """Retourne la taille en Mio d'un répertoire HDFS."""    out = subprocess.run(["hdfs", "dfs", "-du", "-s", chemin],                         capture_output=True, text=True).stdout    return round(int(out.split()[0]) / 1024**2, 1)formats = ["csv", "csv_gz", "json", "avro", "parquet_snappy", "parquet_zstd"]tailles = {f: taille_hdfs(f"{BASE}/{f}") for f in formats}ref = tailles["json"]print(f"{'format':18s} {'Mio':>9s} {'ratio vs json':>15s}")for f in formats:    print(f"{f:18s} {tailles[f]:9.1f} {ref/tailles[f]:14.1f}x")

### Q1 *(2 pts)* — À partir du tableau ci-dessus :- **a.** Quel format est le plus compact ? De combien par rapport à JSON ?- **b.** `csv.gz` et `parquet-snappy` ont des tailles proches. Pourtant gzip compresse  **mieux** que Snappy. Comment expliquez-vous que Parquet fasse aussi bien avec un  codec moins performant ?*(rédigez ici)*

### Q2 *(2 pts)* — Le format `json` est plus volumineux que `csv`. Pourquoi ?Quelle information JSON porte-t-il que CSV ne porte pas, et ce surcoût vousparaît-il justifié pour du stockage de masse ?*(rédigez ici)*

---# Exercice 3 — La splittabilité  *(4 points)***Objectif.** Démontrer, et non croire sur parole, qu'un fichier gzip ne peut pasêtre découpé.

In [ ]:
# 3.1 — Réécrire chaque format en UN SEUL fichier, pour isoler l'effetUNI = f"{BASE}/unique"!hdfs dfs -rm -r -f -skipTrash {UNI}un = df.coalesce(1)un.write.mode("overwrite").option("header", True).csv(f"{UNI}/csv")un.write.mode("overwrite").option("header", True).option("compression", "gzip") \  .csv(f"{UNI}/csv_gz")un.write.mode("overwrite").parquet(f"{UNI}/parquet")!hdfs dfs -ls -h {UNI}/csv {UNI}/csv_gz {UNI}/parquet | grep -v '^Found'

In [ ]:
# 3.2 — À VOUS : combien de partitions Spark crée-t-il pour chacun ?#          Complétez les trois lectures, puis affichez getNumPartitions().for nom in ["csv", "csv_gz", "parquet"]:    ...

### Q3 *(2 pts)* — Le fichier `csv_gz` ne donne qu'**une seule partition**, alors que`csv` et `parquet` en donnent plusieurs, pour un contenu identique.- **a.** Expliquez le mécanisme.- **b.** Parquet est lui aussi compressé. Pourquoi reste-t-il splittable ?*(rédigez ici)*

In [ ]:
# 3.3 — Mesurer la conséquence : temps de lecture complètefor nom in ["csv", "csv_gz", "parquet"]:    t0 = time.time()    if nom == "parquet":        n = spark.read.parquet(f"{UNI}/{nom}").count()    else:        n = spark.read.option("header", True).csv(f"{UNI}/{nom}").count()    print(f"{nom:10s} {n:>10,d} lignes en {time.time()-t0:6.1f} s")

### Q4 *(2 pts)* — Rapprochez ces temps du nombre de partitions de la questionprécédente. L'écart est-il proportionnel au nombre de cœurs de votre machine ?Sinon, quelle autre cause intervient ?*(rédigez ici)*

---# Exercice 4 — Projection et pushdown  *(5 points)***Objectif.** Mesurer séparément les deux gains du format colonne : ne lire que lescolonnes utiles, et n'ouvrir que les blocs utiles.

In [ ]:
# 4.1 — Une requête réaliste : trois colonnes, un filtre sélectifCOL_FILTRE = "montant" if FILIERE == "if" else "ms_mise_en_tampon"SEUIL      = 5000      if FILIERE == "if" else 3000COLONNES   = (["id_transaction", "montant", "pays_transaction"] if FILIERE == "if"              else ["id_evenement", "ms_mise_en_tampon", "pays"])def mesurer(chemin, lecteur):    t0 = time.time()    n = (lecteur(chemin).select(*COLONNES)         .filter(f"{COL_FILTRE} > {SEUIL}").count())    return round(time.time() - t0, 2), nt_csv, n1 = mesurer(f"{BASE}/csv",                    lambda p: spark.read.option("header", True)                                   .schema(schema).csv(p))t_par, n2 = mesurer(f"{BASE}/parquet_snappy", lambda p: spark.read.parquet(p))print(f"csv     : {t_csv:6.2f} s   ({n1:,} lignes)")print(f"parquet : {t_par:6.2f} s   ({n2:,} lignes)")print(f"gain    : {t_csv/t_par:.1f}x")

In [ ]:
# 4.2 — Lire le plan d'exécution : que Spark a-t-il poussé au lecteur ?(spark.read.parquet(f"{BASE}/parquet_snappy")     .select(*COLONNES)     .filter(f"{COL_FILTRE} > {SEUIL}")     .explain(mode="formatted"))

### Q5 *(3 pts)* — Dans le plan ci-dessus, repérez les deux lignes suivantes etrecopiez-les :- **`ReadSchema`** — quelles colonnes Spark a-t-il demandé au lecteur ?- **`PushedFilters`** — quel filtre a été transmis au lecteur Parquet ?Puis répondez : ces deux mécanismes portent des noms différents parce qu'ils agissentsur deux axes différents. Lesquels ?*(rédigez ici)*

In [ ]:
# 4.3 — Isoler le gain de projection : lire TOUTES les colonnes, même filtret0 = time.time()n = (spark.read.parquet(f"{BASE}/parquet_snappy")     .filter(f"{COL_FILTRE} > {SEUIL}").count())t_toutes = round(time.time() - t0, 2)print(f"3 colonnes  : {t_par:6.2f} s")print(f"10 colonnes : {t_toutes:6.2f} s")print(f"gain de la projection seule : {t_toutes/t_par:.1f}x")

### Q6 *(2 pts)* — Le gain total mesuré en 4.1 se décompose en deux facteurs.Calculez-les séparément, et vérifiez que leur produit est cohérent avec le gain global.*(rédigez ici)*

---# Exercice 5 — L'effet du tri  *(3 points)***Objectif.** Montrer que le predicate pushdown ne vaut rien sur des données nontriées — le point que la plupart des ingénieurs ignorent.

In [ ]:
# 5.1 — Écrire la même table, triée sur la colonne filtréet0 = time.time()(df.orderBy(COL_FILTRE).write.mode("overwrite")   .option("compression", "snappy").parquet(f"{BASE}/parquet_trie"))print(f"écriture triée : {time.time()-t0:.1f} s")print("taille non trié :", taille_hdfs(f"{BASE}/parquet_snappy"), "Mio")print("taille trié     :", taille_hdfs(f"{BASE}/parquet_trie"), "Mio")

In [ ]:
# 5.2 — Même requête sur les deux versionst_non, _ = mesurer(f"{BASE}/parquet_snappy", lambda p: spark.read.parquet(p))t_tri, _ = mesurer(f"{BASE}/parquet_trie",   lambda p: spark.read.parquet(p))print(f"non trié : {t_non:6.2f} s")print(f"trié     : {t_tri:6.2f} s")print(f"gain     : {t_non/t_tri:.1f}x")

### Q7 *(3 pts)* —- **a.** Le fichier trié est-il plus petit ? Pourquoi ?- **b.** La requête est-elle plus rapide ? Expliquez par le mécanisme des statistiques  de row group.- **c.** Le tri a un coût à l'écriture. Dans quel cas ce coût est-il rentable, et dans  quel cas ne l'est-il pas ?*(rédigez ici)*

---# Exercice 6 — Synthèse  *(1 point)*Remplissez ce tableau avec **vos** mesures, puis concluez.| Format | Taille (Mio) | Écriture (s) | Requête filtrée (s) | Splittable ||---|---|---|---|---|| CSV | | | | || CSV.gz | | | | || JSON | | | | || Avro | | | | || Parquet snappy | | | | || Parquet zstd | | | | || Parquet trié | | | | |**En trois lignes :** quel format retiendriez-vous pour une table analytique interrogéequotidiennement, et pourquoi ? Citez au moins deux chiffres de vos propres mesures.

In [ ]:
# Nettoyage (optionnel)# !hdfs dfs -rm -r -skipTrash {BASE}spark.stop()print("Session fermée.")

---## Avant de rendre- [ ] Toutes les cellules exécutées dans l'ordre, sorties visibles.- [ ] Les sept questions rédigées et justifiées.- [ ] Le tableau de synthèse rempli avec **vos** mesures.- [ ] Notebook exporté en HTML et déposé sur l'espace de cours.